In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import random
import os

In [0]:
# Схемы для данных
employees_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", DateType(), True),
    StructField("city", StringType(), True),
    StructField("performance_score", IntegerType(), True)
])

sales_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("emp_id", IntegerType(), True),
    StructField("product_category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("sale_date", TimestampType(), True),
    StructField("region", StringType(), True),
    StructField("discount_applied", BooleanType(), True),
    StructField("customer_id", StringType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("loyalty_tier", StringType(), True)
])

print("Генерация данных employees...")
# Генерация данных для employees
departments = ["IT", "HR", "Finance", "Marketing", "Sales", "Operations", "Engineering", "Support"]
cities = ["New York", "London", "Tokyo", "Berlin", "Sydney", "Toronto", "Paris", "Singapore"]

employees_data = []
for i in range(1, 501):
    employees_data.append((
        i,
        f"Employee_{i}",
        random.choice(departments),
        random.randint(45000, 160000),
        datetime(2018, 1, 1) + timedelta(days=random.randint(0, 2190)),
        random.choice(cities),
        random.randint(1, 100)
    ))

employees_df = spark.createDataFrame(employees_data, employees_schema)
print(f"Employees создан: {employees_df.count()} строк")

print("Генерация данных customers...")
# Генерация данных для customers
segments = ["Enterprise", "SMB", "Startup", "Individual"]
tiers = ["Bronze", "Silver", "Gold", "Platinum"]
countries = ["USA", "UK", "Germany", "Japan", "Australia", "Canada", "France", "Brazil", "India", "China"]

customers_data = []
for i in range(1, 201):
    customers_data.append((
        f"CUST{5000 + i}",
        f"Customer_{i}",
        random.choice(segments),
        random.choice(countries),
        datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1460)),
        random.choice(tiers)
    ))

customers_df = spark.createDataFrame(customers_data, customers_schema)
print(f"Customers создан: {customers_df.count()} строк")

print("Генерация данных sales...")
# Генерация данных для sales
products = ["Electronics", "Clothing", "Home", "Books", "Sports", "Beauty", "Food", "Toys"]
regions = ["North America", "Europe", "Asia", "South America", "Africa", "Oceania"]

sales_data = []
for i in range(1, 1001):
    # Генерируем amount
    amount = random.uniform(50.0, 5000.0)
    amount = float(f"{amount:.2f}")
    
    # Создаем datetime
    total_seconds = random.randint(0, 365*24*60*60)
    sale_datetime = datetime(2023, 1, 1) + timedelta(seconds=total_seconds)
    
    sales_data.append((
        f"TXN{10000 + i}",
        random.randint(1, 500),
        random.choice(products),
        amount,
        sale_datetime,
        random.choice(regions),
        random.choice([True, False]),
        f"CUST{5000 + random.randint(1, 200)}"
    ))

sales_df = spark.createDataFrame(sales_data, sales_schema)
print(f"Sales создан: {sales_df.count()} строк")

# Создаем временные представления
employees_df.createOrReplaceTempView("employees")
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")

print("Временные представления созданы!")
print("Доступные представления: employees, sales, customers")

# Покажем немного данных для проверки
print("Пример данных employees:")
employees_df.show(5)

print("Пример данных sales:")
sales_df.show(5)

print("Пример данных customers:")
customers_df.show(5)

In [0]:
sql_query_1 = """
SELECT 
    department,
    COUNT(*) as employee_count,
    ROUND(AVG(salary), 2) as avg_salary,
    MAX(salary) as max_salary,
    MIN(salary) as min_salary
FROM employees
WHERE salary > 50000 AND performance_score > 70
GROUP BY department
HAVING COUNT(*) >= 10
ORDER BY avg_salary DESC
"""

spark_query_1 = employees_df.filter((col('salary') > 50000) & (col('performance_score') > 70))\
    .groupBy('department')\
    .agg(
        count('*').alias('employee_count'),
        round(avg('salary'), 2).alias('avg_salary'),
        max('salary').alias('max_salary'),
        min('salary').alias('min_salary')
    ).filter(col('employee_count') >= 10)\
    .orderBy(col('avg_salary').desc())

spark.sql(sql_query_1).show(1, truncate=False, vertical=True)
spark_query_1.show(1, truncate=False, vertical=True)

In [0]:
sql_query_2 = """
SELECT 
    e.emp_id,
    e.name,
    e.department,
    e.city,
    s.transaction_id,
    s.product_category,
    s.amount,
    DATE(s.sale_date) as sale_date,
    s.region

FROM employees e
INNER JOIN sales s ON e.emp_id = s.emp_id
WHERE s.amount > 1000
  AND e.department IN ('Sales', 'Marketing')
  AND s.discount_applied = TRUE
  AND YEAR(s.sale_date) = 2023
ORDER BY s.amount DESC
LIMIT 15
"""

spark_query_2 = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'inner')\
    .filter((col('s.amount') > 1000) & (col('e.department').isin(['Sales', 'Marketing'])) & (col('s.discount_applied') == 'TRUE') & (year(col('s.sale_date')) == 2023))\
    .select(
        'e.emp_id',
        'e.name',
        'e.department',
        'e.city',
        's.transaction_id',
        's.product_category',
        's.amount',
        to_date(col('s.sale_date')).alias('sale_date'),
        's.region'
    ).orderBy(col('s.amount').desc()).limit(15)

spark.sql(sql_query_2).show(1, truncate=False, vertical=True)
spark_query_2.show(1, truncate=False, vertical=True)

In [0]:
sql_query_3 = """
SELECT 
    customer_id,
    customer_name,
    segment,
    country,
    loyalty_tier,
    signup_date,
    DATEDIFF(CURRENT_DATE(), signup_date) as days_as_customer,
    CASE 
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 1095 THEN 'Loyal (3+ years)'
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 730 THEN 'Established (2-3 years)'
        WHEN DATEDIFF(CURRENT_DATE(), signup_date) > 365 THEN 'Growing (1-2 years)'
        ELSE 'New (<1 year)'
    END as customer_tenure_category,
    RANK() OVER (PARTITION BY country ORDER BY signup_date) as earliest_signup_rank,
    COUNT(*) OVER (PARTITION BY country, segment) as segment_count_in_country
FROM customers
WHERE loyalty_tier IN ('Gold', 'Platinum')
ORDER BY country, days_as_customer DESC
"""

spark_query_3 = customers_df.filter(col('loyalty_tier').isin(['Gold', 'Platinum']))\
    .select(
        'customer_id',
        'customer_name',
        'segment',
        'country',
        'loyalty_tier',
        'signup_date',
        datediff(current_date(), col('signup_date')).alias('days_as_customer'),
        when(col('days_as_customer') > 1095, 'Loyal (3+ years)')\
        .when(col('days_as_customer') > 730, 'Established (2-3 years)')\
        .when(col('days_as_customer') > 365, 'Growing (1-2 years)')\
        .otherwise('New (<1 year)').alias('customer_tenure_category'),
        rank().over(Window.partitionBy('country').orderBy('signup_date')).alias('earliest_signup_rank'),
        count('*').over(Window.partitionBy('country', 'segment')).alias('segment_count_in_country')
    ).orderBy('country', col('days_as_customer').desc())

spark.sql(sql_query_3).show(1, truncate=False, vertical=True)
spark_query_3.show(1, truncate=False, vertical=True)

In [0]:
sql_query_4 = """
SELECT 
    c.country,
    c.segment,
    c.loyalty_tier,
    e.department,
    s.product_category,
    COUNT(DISTINCT s.transaction_id) as total_transactions,
    SUM(s.amount) as total_revenue,
    ROUND(AVG(s.amount), 2) as avg_transaction_value,
    COUNT(DISTINCT e.emp_id) as unique_employees,
    COUNT(DISTINCT c.customer_id) as unique_customers,
    SUM(CASE WHEN s.discount_applied THEN 1 ELSE 0 END) as discounted_transactions,
    ROUND(SUM(CASE WHEN s.discount_applied THEN s.amount ELSE 0 END) * 100.0 / SUM(s.amount), 2) as discount_revenue_percentage
FROM customers c
JOIN sales s ON c.customer_id = s.customer_id
JOIN employees e ON s.emp_id = e.emp_id
WHERE YEAR(s.sale_date) = 2023
    AND c.country IN ('USA', 'UK', 'Germany', 'Japan')
GROUP BY c.country, c.segment, c.loyalty_tier, e.department, s.product_category
HAVING total_transactions >= 5
ORDER BY total_revenue DESC
"""

spark_query_4 = customers_df.alias('c').join(sales_df.alias('s'), 'customer_id', 'inner')\
    .join(employees_df.alias('e'), 'emp_id', 'inner')\
    .where((year(col('s.sale_date')) == 2023) & (col('c.country').isin(['USA', 'UK', 'Germany', 'Japan'])))\
    .groupBy('c.country', 'c.segment', 'c.loyalty_tier', 'e.department', 's.product_category')\
    .agg(
        countDistinct('s.transaction_id').alias('total_transactions'),
        sum('s.amount').alias('total_revenue'),
        round(avg('s.amount'),2).alias('avg_transaction_value'),
        countDistinct('e.emp_id').alias('unique_employees'),
        countDistinct('c.customer_id').alias('unique_customers'),
        sum(when(col('s.discount_applied') == 'TRUE',1).otherwise(0)).alias('discounted_transactions'),
        round(sum(when(col('s.discount_applied') == 'TRUE', col('s.amount')).otherwise(0)) * 100.0 / col('total_revenue'), 2).alias('discount_revenue_percentage')
    ).filter(col('total_transactions') >= 5)\
    .orderBy(col('total_revenue').desc())

spark.sql(sql_query_4).show(1, truncate=False, vertical=True)
spark_query_4.show(1, truncate=False, vertical=True)

In [0]:
sql_query_5 = """
WITH employee_stats AS (
    SELECT 
        e.emp_id,
        e.name,
        e.department,
        e.city,
        e.salary,
        e.performance_score,
        COUNT(s.transaction_id) as total_sales,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        MAX(s.amount) as max_sale_amount
    FROM employees e
    LEFT JOIN sales s ON e.emp_id = s.emp_id
    GROUP BY e.emp_id, e.name, e.department, e.city, e.salary, e.performance_score
)
SELECT 
    emp_id,
    name,
    department,
    city,
    salary,
    performance_score,
    total_sales,
    total_revenue,
    ROUND(avg_sale_amount, 2) as avg_sale_amount,
    max_sale_amount,
    ROUND(total_revenue / NULLIF(salary, 0), 2) as revenue_to_salary_ratio,
    RANK() OVER (PARTITION BY department ORDER BY total_revenue DESC) as dept_revenue_rank,
    CASE 
        WHEN total_revenue > 100000 THEN 'Top Performer'
        WHEN total_revenue > 50000 THEN 'Good Performer'
        WHEN total_revenue > 0 THEN 'Average Performer'
        ELSE 'No Sales'
    END as performance_category
FROM employee_stats
WHERE total_sales > 0
ORDER BY department, total_revenue DESC
"""

emp_stts = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'left')\
    .groupBy('e.emp_id', 'e.name', 'e.department', 'e.city', 'e.salary', 'e.performance_score')\
    .agg(
        countDistinct('s.transaction_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        max('s.amount').alias('max_sale_amount'))
    
spark_query_5 = emp_stts.filter(col('total_sales') > 0)\
    .select(
        'emp_id',
        'name',
        'department',
        'city',
        'salary',
        'performance_score',
        'total_sales',
        'total_revenue',
        round(col('avg_sale_amount'), 2).alias('avg_sale_amount'),
        'max_sale_amount',
        round(col('total_revenue') / nullif(col('salary'), lit(0)), 2).alias('revenue_to_salary_ratio'),
        rank().over(Window.partitionBy('department').orderBy(col('total_revenue').desc())).alias('dept_revenue_rank'),
        when(col('total_revenue') > 100000, 'Top Performer')\
        .when(col('total_revenue') > 50000, 'Good Performer')\
        .when(col('total_revenue') > 0, 'Average Performer')\
        .otherwise('No Sales').alias('performance_category')
    ).orderBy('department', col('total_revenue').desc())

spark.sql(sql_query_5).show(5, truncate=False, vertical=True)
spark_query_5.show(3, truncate=False, vertical=True)

In [0]:
sql_query_6 = """
WITH monthly_sales AS (
    SELECT 
        e.department,
        s.product_category,
        DATE_TRUNC('month', s.sale_date) as sale_month,
        COUNT(*) as monthly_transactions,
        SUM(s.amount) as monthly_revenue,
        AVG(s.amount) as avg_transaction_value,
        COUNT(DISTINCT s.customer_id) as unique_customers
    FROM sales s
    JOIN employees e ON s.emp_id = e.emp_id
    WHERE s.sale_date >= '2023-01-01'
    GROUP BY e.department, s.product_category, DATE_TRUNC('month', s.sale_date)
),
sales_trends AS (
    SELECT 
        *,
        LAG(monthly_revenue, 1) OVER (
            PARTITION BY department, product_category 
            ORDER BY sale_month
        ) as prev_month_revenue,
        LAG(monthly_revenue, 3) OVER (
            PARTITION BY department, product_category 
            ORDER BY sale_month
        ) as prev_quarter_revenue,
        AVG(monthly_revenue) OVER (
            PARTITION BY department, product_category 
            ORDER BY sale_month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) as moving_avg_3_months,
        SUM(monthly_revenue) OVER (
            PARTITION BY department, product_category 
            ORDER BY sale_month 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as cumulative_revenue
    FROM monthly_sales
)
SELECT 
    department,
    product_category,
    sale_month,
    monthly_transactions,
    ROUND(monthly_revenue, 2) as monthly_revenue,
    ROUND(avg_transaction_value, 2) as avg_transaction_value,
    unique_customers,
    ROUND(prev_month_revenue, 2) as prev_month_revenue,
    ROUND(
        (monthly_revenue - prev_month_revenue) * 100.0 / NULLIF(prev_month_revenue, 0), 
        2
    ) as mom_growth_pct,
    ROUND(moving_avg_3_months, 2) as moving_avg_3_months,
    ROUND(
        (monthly_revenue - moving_avg_3_months) * 100.0 / NULLIF(moving_avg_3_months, 0), 
        2
    ) as deviation_from_trend_pct,
    ROUND(cumulative_revenue, 2) as cumulative_revenue,
    CASE 
        WHEN monthly_revenue > moving_avg_3_months * 1.15 THEN 'Significant Growth'
        WHEN monthly_revenue > moving_avg_3_months * 1.05 THEN 'Moderate Growth'
        WHEN monthly_revenue < moving_avg_3_months * 0.95 THEN 'Decline'
        ELSE 'Stable'
    END as trend_category
FROM sales_trends
WHERE monthly_transactions >= 3
ORDER BY department, product_category, sale_month
"""

mon_sal = sales_df.alias('s').join(employees_df.alias('e'), 'emp_id', 'inner')\
    .withColumn('sale_month', date_trunc('month', 's.sale_date'))\
    .filter(col('s.sale_date') >= '2023-01-01')\
    .groupBy('e.department', 's.product_category', 'sale_month')\
    .agg(
        count('*').alias('monthly_transactions'),
        sum('s.amount').alias('monthly_revenue'),
        avg('s.amount').alias('avg_transaction_value'),
        countDistinct('s.customer_id').alias('unique_customers'))
    
sal_tren = mon_sal.select(
    '*',
    lag('monthly_revenue',1).over(Window.partitionBy('department', 'product_category').orderBy('sale_month')).alias('prev_month_revenue'),
    lag('monthly_revenue', 3).over(Window.partitionBy('department', 'product_category').orderBy('sale_month')).alias('prev_quarter_revenue'),
    avg('monthly_revenue').over(Window.partitionBy('department', 'product_category').orderBy('sale_month').rowsBetween(-2, Window.currentRow)).alias('moving_avg_3_months'),
    sum('monthly_revenue').over(Window.partitionBy('department', 'product_category').orderBy('sale_month').rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias('cumulative_revenue'))

spark_query_6 = sal_tren.filter(col('monthly_transactions') >= 3)\
    .select(
        'department',
        'product_category',
        'sale_month',
        'monthly_transactions',
        round(col('monthly_revenue'), 2).alias('monthly_revenue'),
        round(col('avg_transaction_value'), 2).alias('avg_transaction_value'),
        'unique_customers',
        round(col('prev_month_revenue'), 2).alias('prev_month_revenue'),
        round((col('monthly_revenue') - col('prev_month_revenue')) * 100.0 / nullif(col('prev_month_revenue'), lit(0)), 2).alias('mom_growth_pct'),
        round(col('moving_avg_3_months'), 2).alias('moving_avg_3_months'),
        round((col('monthly_revenue') - col('moving_avg_3_months')) * 100.0 / nullif(col('moving_avg_3_months'), lit(0)), 2).alias('deviation_from_trend_pct'),
        round(col('cumulative_revenue'), 2).alias('cumulative_revenue'),
        when(col('monthly_revenue') > (col('moving_avg_3_months') * 1.15), 'Significant Growth')\
        .when(col('monthly_revenue') > (col('moving_avg_3_months') * 1.05), 'Moderate Growth')\
        .when(col('monthly_revenue') < (col('moving_avg_3_months') * 0.95), 'Decline')\
        .otherwise('Stable').alias('trend_category')
    ).orderBy('department', 'product_category', 'sale_month')

spark.sql(sql_query_6).show(1, truncate=False, vertical=True)
spark_query_6.show(1, truncate=False, vertical=True)

In [0]:
sql_query_7 = """
WITH department_efficiency AS (
    SELECT 
        e.department,
        COUNT(DISTINCT e.emp_id) as total_employees,
        COUNT(DISTINCT s.emp_id) as active_sellers,
        COUNT(DISTINCT s.transaction_id) as total_transactions,
        SUM(s.amount) as total_revenue,
        AVG(e.salary) as avg_salary,
        SUM(e.salary) as total_salary_cost,
        COUNT(DISTINCT s.product_category) as unique_products,
        COUNT(DISTINCT s.region) as regions_covered,
        COUNT(DISTINCT s.customer_id) as unique_customers
    FROM employees e
    LEFT JOIN sales s ON e.emp_id = s.emp_id
    GROUP BY e.department
),
department_metrics AS (
    SELECT 
        *,
        active_sellers * 100.0 / total_employees as active_seller_rate,
        total_revenue / NULLIF(total_salary_cost, 0) as roi_ratio,
        total_revenue / NULLIF(total_employees, 0) as revenue_per_employee,
        total_transactions / NULLIF(total_employees, 0) as transactions_per_employee,
        unique_customers / NULLIF(total_employees, 0) as customers_per_employee,
        PERCENT_RANK() OVER (ORDER BY total_revenue) as revenue_percentile,
        PERCENT_RANK() OVER (ORDER BY total_revenue / NULLIF(total_employees, 0)) as efficiency_percentile
    FROM department_efficiency
)
SELECT 
    department,
    total_employees,
    active_sellers,
    ROUND(active_seller_rate, 2) as active_seller_pct,
    total_transactions,
    ROUND(total_revenue, 2) as total_revenue,
    ROUND(avg_salary, 2) as avg_salary,
    ROUND(total_salary_cost, 2) as total_salary_cost,
    unique_products,
    regions_covered,
    unique_customers,
    ROUND(roi_ratio, 2) as roi_ratio,
    ROUND(revenue_per_employee, 2) as revenue_per_emp,
    ROUND(transactions_per_employee, 2) as transactions_per_emp,
    ROUND(customers_per_employee, 2) as customers_per_emp,
    ROUND(revenue_percentile * 100, 2) as revenue_percentile_pct,
    ROUND(efficiency_percentile * 100, 2) as efficiency_percentile_pct,
    CASE 
        WHEN revenue_percentile >= 0.8 AND efficiency_percentile >= 0.8 THEN 'High Revenue & High Efficiency'
        WHEN revenue_percentile >= 0.8 THEN 'High Revenue & Low Efficiency'
        WHEN efficiency_percentile >= 0.8 THEN 'Low Revenue & High Efficiency'
        ELSE 'Low Revenue & Low Efficiency'
    END as performance_quadrant,
    CASE 
        WHEN roi_ratio > 2 THEN 'Excellent ROI'
        WHEN roi_ratio > 1 THEN 'Good ROI'
        WHEN roi_ratio > 0.5 THEN 'Moderate ROI'
        ELSE 'Poor ROI'
    END as roi_category
FROM department_metrics
ORDER BY roi_ratio DESC
"""

dep_eff = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'left')\
    .groupBy('e.department')\
    .agg(
        countDistinct('e.emp_id').alias('total_employees'),
        countDistinct('s.emp_id').alias('active_sellers'),
        countDistinct('s.transaction_id').alias('total_transactions'),
        sum('s.amount').alias('total_revenue'),
        avg('e.salary').alias('avg_salary'),
        sum('e.salary').alias('total_salary_cost'),
        countDistinct('s.product_category').alias('unique_products'),
        countDistinct('s.region').alias('regions_covered'),
        countDistinct('s.customer_id').alias('unique_customers'))
    
dep_metr = dep_eff.select(
    '*',
    (col('active_sellers') * 100 / col('total_employees')).alias('active_seller_rate'),
    (col('total_revenue') / nullif(col('total_salary_cost'), lit(0))).alias('roi_ratio'),
    (col('total_revenue') / nullif(col('total_employees'), lit(0))).alias('revenue_per_employee'),
    (col('total_transactions') / nullif(col('total_employees'), lit(0))).alias('transactions_per_employee'),
    (col('unique_customers') / nullif(col('total_employees'), lit(0))).alias('customers_per_employee'),
    percent_rank().over(Window.orderBy('total_revenue')).alias('revenue_percentile'),
    percent_rank().over(Window.orderBy(col('total_revenue') / nullif(col('total_employees'), lit(0)))).alias('efficiency_percentile'))

spark_query_7 = dep_metr.select(
    'department',
    'total_employees',
    'active_sellers',
    round(col('active_seller_rate'), 2).alias('active_seller_pct'),
    'total_transactions',
    round(col('total_revenue'), 2).alias('total_revenue'),
    round(col('avg_salary'), 2).alias('avg_salary'),
    round(col('total_salary_cost'), 2).alias('total_salary_cost'),
    'unique_products',
    'regions_covered',
    'unique_customers',
    round(col('roi_ratio'), 2).alias('roi_ratio'),
    round(col('revenue_per_employee'), 2).alias('revenue_per_emp'),
    round(col('transactions_per_employee'), 2).alias('transactions_per_emp'),
    round(col('customers_per_employee'), 2).alias('customers_per_emp'),
    round(col('revenue_percentile') * 100, 2).alias('revenue_percentile_pct'),
    round(col('efficiency_percentile') * 100, 2).alias('efficiency_percentile_pct'),
    when((col('revenue_percentile') >= 0.8) & (col('efficiency_percentile') >= 0.8), 'High Revenue & High Efficiency')\
    .when((col('revenue_percentile') >= 0.8), 'High Revenue & Low Efficiency')\
    .when(col('efficiency_percentile') >= 0.8, 'Low Revenue & High Efficiency')\
    .otherwise('Low_Revenue & Low_Efficiency').alias('performance_quadrant'),
    when(col('roi_ratio') > 2, 'Excellent ROI')\
    .when(col('roi_ratio') > 1, 'Good ROI')\
    .when(col('roi_ratio') > 0.5, 'Moderate ROI')\
    .otherwise('Poor ROI').alias('roi_category')
).orderBy(col('roi_ratio').desc())

spark.sql(sql_query_7).show(1, truncate=False, vertical=True)
spark_query_7.show(1, truncate=False, vertical=True)

In [0]:
sql_query_8 = """
WITH customer_rfm AS (
    SELECT 
        c.customer_id,
        c.customer_name,
        c.segment,
        c.country,
        c.loyalty_tier,
        c.signup_date,
        -- Recency: Days since last purchase
        DATEDIFF(CURRENT_DATE(), MAX(s.sale_date)) as recency_days,
        -- Frequency: Number of purchases
        COUNT(DISTINCT s.transaction_id) as frequency,
        -- Monetary: Total spending
        SUM(s.amount) as monetary,
        -- Additional metrics
        COUNT(DISTINCT s.product_category) as product_categories_purchased,
        COUNT(DISTINCT s.emp_id) as salespeople_interacted_with,
        AVG(s.amount) as avg_transaction_value,
        MAX(s.amount) as max_transaction_value,
        MIN(s.sale_date) as first_purchase_date,
        MAX(s.sale_date) as last_purchase_date,
        DATEDIFF(MAX(s.sale_date), MIN(s.sale_date)) as customer_lifespan_days
    FROM customers c
    JOIN sales s ON c.customer_id = s.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment, c.country, c.loyalty_tier, c.signup_date
),
rfm_scores AS (
    SELECT 
        *,
        -- RFM Scoring (1-5 scale, 5 being best)
        NTILE(5) OVER (ORDER BY recency_days DESC) as recency_score,  -- Higher recency_days is worse
        NTILE(5) OVER (ORDER BY frequency) as frequency_score,
        NTILE(5) OVER (ORDER BY monetary) as monetary_score,
        (NTILE(5) OVER (ORDER BY recency_days DESC) + 
         NTILE(5) OVER (ORDER BY frequency) + 
         NTILE(5) OVER (ORDER BY monetary)) as rfm_total_score
    FROM customer_rfm
    WHERE frequency >= 2  -- Only customers with at least 2 purchases
),
customer_segments AS (
    SELECT 
        *,
        CASE 
            WHEN recency_score >= 4 AND frequency_score >= 4 AND monetary_score >= 4 THEN 'Champions'
            WHEN recency_score >= 3 AND frequency_score >= 3 AND monetary_score >= 3 THEN 'Loyal Customers'
            WHEN recency_score >= 4 THEN 'New Customers'
            WHEN recency_score <= 2 AND frequency_score <= 2 AND monetary_score <= 2 THEN 'At Risk'
            WHEN recency_score <= 2 THEN 'Hibernating'
            ELSE 'Regular Customers'
        END as rfm_segment,
        CASE 
            WHEN monetary > 10000 THEN 'VIP'
            WHEN monetary > 5000 THEN 'Premium'
            WHEN monetary > 1000 THEN 'Regular'
            ELSE 'Budget'
        END as spending_tier,
        monetary / NULLIF(customer_lifespan_days, 0) as daily_spending_rate
    FROM rfm_scores
)
SELECT 
    customer_id,
    customer_name,
    segment,
    country,
    loyalty_tier,
    recency_days,
    frequency,
    ROUND(monetary, 2) as total_spent,
    recency_score,
    frequency_score,
    monetary_score,
    rfm_total_score,
    rfm_segment,
    spending_tier,
    product_categories_purchased,
    salespeople_interacted_with,
    ROUND(avg_transaction_value, 2) as avg_transaction_value,
    ROUND(max_transaction_value, 2) as max_transaction_value,
    ROUND(daily_spending_rate, 2) as daily_spending_rate,
    CASE 
        WHEN rfm_segment = 'Champions' THEN 'Focus on retention and upselling'
        WHEN rfm_segment = 'Loyal Customers' THEN 'Reward programs and loyalty benefits'
        WHEN rfm_segment = 'New Customers' THEN 'Welcome series and onboarding'
        WHEN rfm_segment = 'At Risk' THEN 'Win-back campaigns and special offers'
        WHEN rfm_segment = 'Hibernating' THEN 'Re-engagement campaigns'
        ELSE 'Regular communication and promotions'
    END as marketing_recommendation
FROM customer_segments
ORDER BY rfm_total_score DESC, monetary DESC
"""

cust_rfm = customers_df.alias('c').join(sales_df.alias('s'), 'customer_id', 'inner')\
    .groupBy('c.customer_id', 'c.customer_name', 'c.segment', 'c.country', 'c.loyalty_tier', 'c.signup_date')\
    .agg(
        datediff(current_date(), max('s.sale_date')).alias('recency_days'),
        countDistinct('s.transaction_id').alias('frequency'),
        sum('s.amount').alias('monetary'),
        countDistinct('s.product_category').alias('product_categories_purchased'),
        countDistinct('s.emp_id').alias('salespeople_interacted_with'),
        avg('s.amount').alias('avg_transaction_value'),
        max('s.amount').alias('max_transaction_value'),
        min('s.sale_date').alias('first_purchase_date'),
        max('s.sale_date').alias('lat_purchase_date'),
        datediff(max('s.sale_date'), min('s.sale_date')).alias('customer_lifespan_days')
    )

rfm_scores = cust_rfm.filter(col('frequency') >= 2)\
    .select(
        '*',
        ntile(5).over(Window.orderBy(col('recency_days').desc())).alias('recency_score'),
        ntile(5).over(Window.orderBy('frequency')).alias('frequency_score'),
        ntile(5).over(Window.orderBy('monetary')).alias('monetary_score'),
        (col('recency_score') + col('frequency_score') + col('monetary_score')).alias('rfm_total_score')
    )

cust_seg = rfm_scores.select(
    '*',
    when((col('recency_score') >= 4) & (col('frequency_score') >= 4) & (col('monetary_score') >= 4), 'Champions')\
    .when((col('recency_score') >= 3) & (col('frequency_score') >= 3) & (col('monetary_score') >= 3), 'Loyal Customers')\
    .when((col('recency_score') >= 4), 'New Customers')\
    .when((col('recency_score') <= 2) & (col('frequency_score') <= 2) & (col('monetary_score') <= 2), 'At Risk')\
    .when((col('recency_score') <= 2), 'Hibernating')\
    .otherwise('Regular Customers').alias('rfm_segment'),
    when(col('monetary') > 10000, 'VIP')\
    .when(col('monetary') > 5000, 'Premium')\
    .when(col('monetary') > 1000, 'Regular')\
    .otherwise('Budget').alias('spending_tier'),
    round(col('monetary') / nullif(col('customer_lifespan_days'), lit(0)), 2).alias('daily_spending_rate'))

spark_query_8 = cust_seg.select(
    'customer_id',
    'customer_name',
    'segment',
    'country',
    'loyalty_tier',
    'recency_days',
    'frequency',
    round(col('monetary'), 2).alias('total_spent'),
    'recency_score',
    'frequency_score',
    'monetary_score',
    'rfm_total_score',
    'rfm_segment',
    'spending_tier',
    'product_categories_purchased',
    'salespeople_interacted_with',
    round(col('avg_transaction_value') ,2).alias('avg_transaction_value'),
    round(col('max_transaction_value'), 2).alias('max_transaction_value'),
    round(col('daily_spending_rate'), 2).alias('daily_spending_rate'),
    when(col('rfm_segment') == 'Champions', 'Focus on retention and upselling')\
    .when(col('rfm_segment') == 'Loyal Customers', 'Reward programs and loyalty benefits')\
    .when(col('rfm_segment') == 'New Customers', 'Welcome series and onboarding')\
    .when(col('rfm_segment') == 'At Risk', 'Win-back campaigns and special offers')\
    .when(col('rfm_segment') == 'Hibernating', 'Re-engagement campaigns')\
    .otherwise('Regular communication and promotions').alias('marketing_recommendation')
).orderBy(col('rfm_total_score').desc(), col('monetary').desc())

spark.sql(sql_query_8).show(1, truncate=False, vertical=True)
spark_query_8.show(1, truncate=False, vertical=True)

In [0]:
sql_query_9 = """
WITH employee_tenure AS (
    SELECT 
        emp_id,
        name,
        department,
        city,
        salary,
        hire_date,
        performance_score,
        DATEDIFF(CURRENT_DATE(), hire_date) as total_days_employed,
        FLOOR(DATEDIFF(CURRENT_DATE(), hire_date) / 365.25) as full_years_employed
    FROM employees
),
employee_sales_performance AS (
    SELECT 
        e.emp_id,
        COUNT(s.transaction_id) as career_sales,
        SUM(s.amount) as career_revenue,
        AVG(s.amount) as career_avg_sale,
        MAX(s.amount) as career_max_sale,
        COUNT(DISTINCT YEAR(s.sale_date)) as active_years,
        COUNT(DISTINCT s.product_category) as products_sold,
        COUNT(DISTINCT s.region) as regions_worked,
        -- Last year performance
        SUM(CASE WHEN s.sale_date >= DATE_SUB(CURRENT_DATE(), 365) THEN s.amount ELSE 0 END) as last_year_revenue,
        COUNT(CASE WHEN s.sale_date >= DATE_SUB(CURRENT_DATE(), 365) THEN s.transaction_id END) as last_year_sales
    FROM employees e
    LEFT JOIN sales s ON e.emp_id = s.emp_id
    GROUP BY e.emp_id
),
career_progression AS (
    SELECT 
        et.*,
        esp.career_sales,
        esp.career_revenue,
        esp.career_avg_sale,
        esp.career_max_sale,
        esp.active_years,
        esp.products_sold,
        esp.regions_worked,
        esp.last_year_revenue,
        esp.last_year_sales,
        -- Compensation analysis
        et.salary / NULLIF(esp.career_revenue, 0) as salary_to_revenue_ratio,
        -- Peer group comparisons
        AVG(et.salary) OVER (PARTITION BY et.department, et.full_years_employed) as avg_salary_peer_group,
        AVG(esp.career_revenue) OVER (PARTITION BY et.department, et.full_years_employed) as avg_revenue_peer_group,
        -- Percentile rankings
        PERCENT_RANK() OVER (PARTITION BY et.department ORDER BY et.salary) as salary_percentile_dept,
        PERCENT_RANK() OVER (PARTITION BY et.department ORDER BY esp.career_revenue) as revenue_percentile_dept,
        -- Growth trajectory
        esp.last_year_revenue / NULLIF(esp.career_avg_sale, 0) as recent_productivity_ratio
    FROM employee_tenure et
    JOIN employee_sales_performance esp ON et.emp_id = esp.emp_id
    WHERE esp.career_sales > 0
)
SELECT 
    emp_id,
    name,
    department,
    city,
    full_years_employed,
    performance_score,
    ROUND(salary, 2) as salary,
    ROUND(career_revenue, 2) as career_revenue,
    career_sales,
    products_sold,
    regions_worked,
    ROUND(salary_to_revenue_ratio, 4) as cost_effectiveness_ratio,
    ROUND(avg_salary_peer_group, 2) as peer_avg_salary,
    ROUND(avg_revenue_peer_group, 2) as peer_avg_revenue,
    ROUND(salary_percentile_dept * 100, 2) as salary_percentile,
    ROUND(revenue_percentile_dept * 100, 2) as revenue_percentile,
    ROUND(recent_productivity_ratio, 2) as recent_productivity,
    -- Compensation fairness analysis
    CASE 
        WHEN salary_percentile_dept > revenue_percentile_dept + 0.2 THEN 'Potentially Overpaid'
        WHEN salary_percentile_dept < revenue_percentile_dept - 0.2 THEN 'Potentially Underpaid'
        ELSE 'Fairly Compensated'
    END as compensation_fairness,
    -- Career stage and potential
    CASE 
        WHEN full_years_employed < 2 AND revenue_percentile_dept > 0.7 THEN 'High Potential - Early Career'
        WHEN full_years_employed BETWEEN 2 AND 5 AND revenue_percentile_dept > 0.8 THEN 'Rising Star - Mid Career'
        WHEN full_years_employed > 5 AND revenue_percentile_dept > 0.9 THEN 'Peak Performer - Senior'
        WHEN full_years_employed > 5 AND revenue_percentile_dept < 0.3 THEN 'Performance Review Needed'
        ELSE 'Standard Career Progression'
    END as career_stage,
    -- Recommendations
    CASE 
        WHEN compensation_fairness = 'Potentially Underpaid' AND career_stage LIKE 'High Potential%' 
            THEN 'Priority for Raise and Promotion'
        WHEN compensation_fairness = 'Potentially Overpaid' AND career_stage = 'Performance Review Needed' 
            THEN 'Performance Improvement Plan'
        WHEN career_stage LIKE 'Rising Star%' 
            THEN 'Development Program and Increased Responsibilities'
        WHEN recent_productivity > 1.5 
            THEN 'Recent High Performer - Monitor for Promotion'
        ELSE 'Continue Current Development Plan'
    END as hr_recommendation
FROM career_progression
ORDER BY department, career_stage, compensation_fairness DESC
"""

emp_ten = employees_df.select(
    'emp_id',
    'name',
    'department',
    'city',
    'salary',
    'hire_date',
    'performance_score',
    datediff(current_date(), col('hire_date')).alias('total_days_employed'),
    floor(col('total_days_employed') / 365.25).alias('full_years_employed'))

emp_sal_per = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'left')\
    .groupBy('e.emp_id')\
    .agg(
        count('s.transaction_id').alias('career_sales'),
        sum('s.amount').alias('career_revenue'),
        avg('s.amount').alias('career_avg_sale'),
        max('s.amount').alias('career_max_sale'),
        countDistinct(year(col('s.sale_date'))).alias('active_years'),
        countDistinct('s.product_category').alias('products_sold'),
        countDistinct('s.region').alias('regions_worked'),
        sum(when(col('s.sale_date') >= date_sub(current_date(), 365), col('s.amount')).otherwise(0)).alias('last_year_revenue'),
        count(when(col('s.sale_date') >= date_sub(current_date(), 365), col('s.transaction_id')).otherwise(0)).alias('last_year_sales'))
    
car_prog = emp_ten.alias('et').join(emp_sal_per.alias('esp'), 'emp_id', 'inner')\
    .filter(col('esp.career_sales') > 0)\
    .select(
        'et.*',
        'esp.career_sales',
        'esp.career_revenue',
        'esp.career_avg_sale',
        'esp.career_max_sale',
        'esp.active_years',
        'esp.products_sold',
        'esp.regions_worked',
        'esp.last_year_revenue',
        'esp.last_year_sales',
        (col('et.salary') / nullif(col('esp.career_revenue'), lit(0))).alias('salary_to_revenue_ratio'),
        avg('et.salary').over(Window.partitionBy('et.department', 'et.full_years_employed')).alias('avg_salary_peer_group'),
        avg('esp.career_revenue').over(Window.partitionBy('et.department', 'et.full_years_employed')).alias('avg_revenue_peer_group'),
        percent_rank().over(Window.partitionBy('et.department').orderBy('et.salary')).alias('salary_percentile_dept'),
        percent_rank().over(Window.partitionBy('et.department').orderBy('esp.career_revenue')).alias('revenue_percentile_dept'),
        (col('esp.last_year_revenue') / nullif(col('esp.career_avg_sale'), lit(0))).alias('recent_productivity_ratio'))
    
spark_query_9 = car_prog.select(
    'emp_id',
    'name',
    'department',
    'city',
    'full_years_employed',
    'performance_score',
    round(col('salary'), 2).alias('salary'),
    round(col('career_revenue'), 2).alias('career_revenue'),
    'career_sales',
    'products_sold',
    'regions_worked',
    round(col('salary_to_revenue_ratio'), 4).alias('cost_effectiveness_ratio'),
    round(col('avg_salary_peer_group'), 2).alias('peer_avg_salary'),
    round(col('avg_revenue_peer_group'), 2).alias('peer_avg_revenue'),
    round(col('salary_percentile_dept') * 100, 2).alias('salary_percentile'),
    round(col('revenue_percentile_dept') * 100, 2).alias('revenue_percentile'),
    round(col('recent_productivity_ratio'), 2).alias('recent_productivity'),
    when((col('salary_percentile_dept') > (col('revenue_percentile_dept') + 0.2)), 'Potentially Overpaid')\
    .when((col('salary_percentile_dept') < (col('revenue_percentile_dept') - 0.2)), 'Potentially Underpaid')\
    .otherwise('Fairly Compensated').alias('compensation_fairness'),
    when((col('full_years_employed') < 2) & (col('revenue_percentile_dept') > 0.7), 'High Potential - Early Career')\
    .when((col('full_years_employed').between(2, 5)) & (col('revenue_percentile_dept') > 0.8), 'Rising Star - Mid Career')\
    .when((col('full_years_employed') > 5) & (col('revenue_percentile_dept') > 0.9), 'Peak Performer - Senior')\
    .when((col('full_years_employed') > 5) & (col('revenue_percentile_dept') < 0.3), 'Performance Review Needed')\
    .otherwise('Standard Career Progression').alias('career_stage'),
    when((col('compensation_fairness') == 'Potentially Underpaid') & (col('career_stage') == 'High Potential%'), 'Priority for Raise and Promotion')\
    .when((col('compensation_fairness') == 'Potentially Overpaid') & (col('career_stage') == 'Performance Review Needed'), 'Performance Improvement Plan')\
    .when((col('career_stage') == 'Rising Star%'), 'Development Program and Increased Responsibilities')\
    .when((col('recent_productivity') > 1.5), 'Recent High Performer - Monitor for Promotion')\
    .otherwise('Continue Current Development Plan').alias('hr_recommendation'))\
    .orderBy('department', 'career_stage', col('compensation_fairness').desc())

spark.sql(sql_query_9).show(1, truncate=False, vertical=True)
spark_query_9.show(1, truncate=False, vertical=True)

In [0]:
sql_query_10 = """
WITH product_region_performance AS (
    SELECT 
        s.product_category,
        s.region,
        e.department,
        DATE_TRUNC('quarter', s.sale_date) as sale_quarter,
        COUNT(*) as transaction_count,
        SUM(s.amount) as quarterly_revenue,
        AVG(s.amount) as avg_transaction_value,
        COUNT(DISTINCT s.emp_id) as unique_sellers,
        COUNT(DISTINCT s.customer_id) as unique_customers,
        SUM(CASE WHEN s.discount_applied THEN 1 ELSE 0 END) as discounted_transactions,
        SUM(CASE WHEN s.discount_applied THEN s.amount ELSE 0 END) as discounted_revenue
    FROM sales s
    JOIN employees e ON s.emp_id = e.emp_id
    WHERE s.sale_date >= '2023-01-01'
    GROUP BY s.product_category, s.region, e.department, DATE_TRUNC('quarter', s.sale_date)
),
product_growth_analysis AS (
    SELECT 
        *,
        -- Growth metrics
        LAG(quarterly_revenue, 1) OVER (
            PARTITION BY product_category, region, department 
            ORDER BY sale_quarter
        ) as prev_quarter_revenue,
        -- Market share analysis
        quarterly_revenue * 100.0 / SUM(quarterly_revenue) OVER (
            PARTITION BY region, department, sale_quarter
        ) as pct_of_region_dept_revenue,
        -- Performance rankings
        RANK() OVER (
            PARTITION BY region, department, sale_quarter 
            ORDER BY quarterly_revenue DESC
        ) as revenue_rank,
        DENSE_RANK() OVER (
            PARTITION BY region, department, sale_quarter 
            ORDER BY avg_transaction_value DESC
        ) as avg_value_rank,
        -- Seller concentration analysis
        unique_sellers * 100.0 / SUM(unique_sellers) OVER (
            PARTITION BY product_category, sale_quarter
        ) as seller_concentration_pct
    FROM product_region_performance
    WHERE transaction_count >= 5
),
product_strategy_categories AS (
    SELECT 
        *,
        ROUND((quarterly_revenue - prev_quarter_revenue) * 100.0 / NULLIF(prev_quarter_revenue, 0), 2) as qoq_growth_pct,
        -- Product strategy categorization
        CASE 
            WHEN revenue_rank <= 2 AND avg_value_rank <= 2 THEN 'Star Product'
            WHEN revenue_rank <= 2 THEN 'High Revenue - Moderate Value'
            WHEN avg_value_rank <= 2 THEN 'Low Revenue - High Value'
            WHEN qoq_growth_pct > 20 THEN 'Emerging Product'
            ELSE 'Standard Product'
        END as product_strategy_category,
        -- Risk analysis
        CASE 
            WHEN seller_concentration_pct > 50 THEN 'High Concentration Risk'
            WHEN seller_concentration_pct > 30 THEN 'Medium Concentration Risk'
            ELSE 'Low Concentration Risk'
        END as concentration_risk,
        -- Discount effectiveness
        discounted_transactions * 100.0 / transaction_count as discount_penetration_pct,
        discounted_revenue * 100.0 / quarterly_revenue as discount_revenue_pct
    FROM product_growth_analysis
)
SELECT 
    product_category,
    region,
    department,
    sale_quarter,
    transaction_count,
    ROUND(quarterly_revenue, 2) as quarterly_revenue,
    ROUND(avg_transaction_value, 2) as avg_transaction_value,
    unique_sellers,
    unique_customers,
    ROUND(pct_of_region_dept_revenue, 2) as market_share_pct,
    revenue_rank,
    avg_value_rank,
    ROUND(qoq_growth_pct, 2) as growth_rate_pct,
    product_strategy_category,
    concentration_risk,
    ROUND(discount_penetration_pct, 2) as discount_usage_pct,
    ROUND(discount_revenue_pct, 2) as discount_revenue_pct,
    -- Strategic recommendations
    CASE 
        WHEN product_strategy_category = 'Star Product' AND concentration_risk = 'Low Concentration Risk' 
            THEN 'Invest & Expand - High Priority'
        WHEN product_strategy_category = 'Star Product' 
            THEN 'Maintain & Diversify Sellers'
        WHEN product_strategy_category = 'Emerging Product' AND growth_rate_pct > 30 
            THEN 'Accelerate Growth - Additional Resources'
        WHEN product_strategy_category = 'Emerging Product' 
            THEN 'Test & Monitor - Controlled Growth'
        WHEN product_strategy_category = 'High Revenue - Moderate Value' AND discount_usage_pct < 10 
            THEN 'Test Strategic Discounts'
        WHEN growth_rate_pct < -10 
            THEN 'Review & Potential Phase-Out'
        ELSE 'Maintain Current Strategy'
    END as strategic_recommendation,
    -- Resource allocation priority
    CASE 
        WHEN product_strategy_category = 'Star Product' THEN 'High Priority'
        WHEN product_strategy_category = 'Emerging Product' AND growth_rate_pct > 25 THEN 'High Priority'
        WHEN revenue_rank <= 3 THEN 'Medium Priority'
        ELSE 'Standard Priority'
    END as resource_priority
FROM product_strategy_categories
ORDER BY sale_quarter, region, department, revenue_rank
"""

prod_reg_per = sales_df.alias('s').join(employees_df.alias('e'), 'emp_id', 'inner')\
    .filter(col('s.sale_date') >= '2023-01-01')\
    .withColumn('sale_quarter', quarter(col('s.sale_date')))\
    .groupBy('s.product_category', 's.region', 'e.department', 'sale_quarter')\
    .agg(
        count('*').alias('transaction_count'),
        sum('s.amount').alias('quarterly_revenue'),
        avg('s.amount').alias('avg_transaction_value'),
        countDistinct('s.emp_id').alias('unique_sellers'),
        countDistinct('s.customer_id').alias('unique_customers'),
        sum(when(col('s.discount_applied'), 1).otherwise(0)).alias('discounted_transactions'),
        sum(when(col('s.discount_applied'), col('s.amount')).otherwise(0)).alias('discounted_revenue'),
    )

prod_gr_an = prod_reg_per.filter(col('transaction_count') >= 5)\
    .select(
        '*',
        lag(('quarterly_revenue'), 1).over(Window.partitionBy('product_category', 'region', 'department').orderBy('sale_quarter')).alias('prev_quarter_revenue'),
        (col('quarterly_revenue') * 100.0 / sum('quarterly_revenue').over(Window.partitionBy('region', 'department', 'sale_quarter'))).alias('pct_of_region_dept_revenue'),
        rank().over(Window.partitionBy('region', 'department', 'sale_quarter').orderBy(col('quarterly_revenue').desc())).alias('revenue_rank'),
        dense_rank().over(Window.partitionBy('region', 'department', 'sale_quarter').orderBy(col('avg_transaction_value').desc())).alias('avg_value_rank'),
        (col('unique_sellers') * 100.0 / sum('unique_sellers').over(Window.partitionBy('product_category', 'sale_quarter'))).alias('seller_concentration_pct')
    )

prod_str_cat = prod_gr_an.select(
    '*',
    round((col('quarterly_revenue') - col('prev_quarter_revenue')) * 100.0 / nullif(col('prev_quarter_revenue'), lit(0)), 2).alias('qoq_growth_pct'),
    when((col('revenue_rank') <= 2) & (col('avg_value_rank') <= 2), 'Star Product')\
            .when((col('revenue_rank') <= 2), 'High Revenue - Moderate Value')\
            .when((col('avg_value_rank') <= 2), 'Low Revenue - High Value')\
            .otherwise('Standard Product').alias('product_strategy_category'),
    when(col('seller_concentration_pct') > 50, 'High Concentration Risk')\
        .when(col('seller_concentration_pct') > 30, 'Medium Concentraion Risk')\
        .otherwise('Low Concentration Risk').alias('concentration_risk'),
    (col('discounted_transactions') * 100.0 / col('transaction_count')).alias('discount_penetration_pct'),
    (col('discounted_revenue') * 100.0 / col('quarterly_revenue')).alias('discount_revenue_pct'))

spark_query_10 = prod_str_cat.select(
    'product_category',
    'region',
    'department',
    'sale_quarter',
    'transaction_count',
    round(col('quarterly_revenue'), 2).alias('quarterly_revenue'),
    round(col('avg_transaction_value'), 2).alias('avg_transaction_value'),
    'unique_sellers',
    'unique_customers',
    round(col('pct_of_region_dept_revenue'), 2).alias('market_share_pct'),
    'revenue_rank',
    'avg_value_rank',
    round(col('qoq_growth_pct'), 2).alias('growth_rate_pct'),
    'product_strategy_category',
    'concentration_risk',
    round(col('discount_penetration_pct'), 2).alias('discount_usage_pct'),
    round(col('discount_revenue_pct'), 2).alias('discount_revenue_pct')
).orderBy('sale_quarter', 'region', 'department', 'revenue_rank')

spark.sql(sql_query_10).show(1, truncate=False, vertical=True)
spark_query_10.show(1, truncate=False, vertical=True)

In [0]:
sql_query = '''
WITH department_stats AS (
    -- Базовая статистика по отделам
    SELECT 
        e.department,
        COUNT(DISTINCT e.emp_id) as total_employees,
        AVG(e.salary) as avg_salary,
        SUM(e.salary) as total_salary_budget,
        AVG(e.performance_score) as avg_performance
    FROM employees e
    GROUP BY e.department
),
employee_sales_performance AS (
    -- Продажи и эффективность сотрудников
    SELECT 
        e.emp_id,
        e.name,
        e.department,
        e.salary,
        e.performance_score,
        e.hire_date,
        COUNT(DISTINCT s.transaction_id) as total_transactions,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        COUNT(DISTINCT s.product_category) as unique_products_sold,
        COUNT(DISTINCT s.region) as regions_covered,
        -- Сезонные продажи
        SUM(CASE WHEN MONTH(s.sale_date) IN (12, 1, 2) THEN s.amount ELSE 0 END) as winter_revenue,
        SUM(CASE WHEN MONTH(s.sale_date) IN (3, 4, 5) THEN s.amount ELSE 0 END) as spring_revenue,
        SUM(CASE WHEN MONTH(s.sale_date) IN (6, 7, 8) THEN s.amount ELSE 0 END) as summer_revenue,
        SUM(CASE WHEN MONTH(s.sale_date) IN (9, 10, 11) THEN s.amount ELSE 0 END) as fall_revenue
    FROM employees e
    LEFT JOIN sales s ON e.emp_id = s.emp_id
    WHERE YEAR(s.sale_date) = 2023 OR s.sale_date IS NULL
    GROUP BY e.emp_id, e.name, e.department, e.salary, e.performance_score, e.hire_date
),
employee_ranked AS (
    -- Ранжирование сотрудников по различным метрикам
    SELECT 
        esp.*,
        ds.avg_salary as department_avg_salary,
        ds.avg_performance as department_avg_performance,
        ds.total_salary_budget,
        -- Ранги внутри отдела
        RANK() OVER (PARTITION BY esp.department ORDER BY esp.total_revenue DESC) as revenue_rank,
        DENSE_RANK() OVER (PARTITION BY esp.department ORDER BY esp.performance_score DESC) as performance_rank,
        PERCENT_RANK() OVER (PARTITION BY esp.department ORDER BY esp.total_revenue) as revenue_percentile,
        -- Сравнение с средними по отделу
        (esp.salary - ds.avg_salary) as salary_diff_from_avg,
        (esp.performance_score - ds.avg_performance) as performance_diff_from_avg,
        -- Эффективность (выручка на зарплату)
        CASE 
            WHEN esp.salary > 0 THEN esp.total_revenue / esp.salary 
            ELSE 0 
        END as roi_ratio,
        -- Стабильность продаж (коэффициент вариации по сезонам)
        CASE 
            WHEN (esp.winter_revenue + esp.spring_revenue + esp.summer_revenue + esp.fall_revenue) > 0 THEN
                SQRT(
                    (POWER(esp.winter_revenue - (esp.total_revenue/4), 2) +
                     POWER(esp.spring_revenue - (esp.total_revenue/4), 2) +
                     POWER(esp.summer_revenue - (esp.total_revenue/4), 2) +
                     POWER(esp.fall_revenue - (esp.total_revenue/4), 2)) / 4
                ) / (esp.total_revenue/4)
            ELSE 0
        END as seasonality_coefficient
    FROM employee_sales_performance esp
    JOIN department_stats ds ON esp.department = ds.department
),
customer_analysis AS (
    -- Анализ клиентской базы сотрудников
    SELECT 
        s.emp_id,
        COUNT(DISTINCT s.customer_id) as unique_customers,
        AVG(CASE WHEN c.loyalty_tier IN ('Gold', 'Platinum') THEN 1 ELSE 0 END) * 100 as premium_customer_percentage,
        COUNT(DISTINCT c.country) as countries_covered,
        -- RFM-метрики для клиентов сотрудника
        AVG(DATEDIFF(CURRENT_DATE, c.signup_date)) as avg_customer_tenure_days,
        COUNT(DISTINCT CASE WHEN c.loyalty_tier IN ('Gold', 'Platinum') THEN s.customer_id END) as premium_customers_count
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    WHERE YEAR(s.sale_date) = 2023
    GROUP BY s.emp_id
),
final_analysis AS (
    -- Финальный анализ с объединением всех данных
    SELECT 
        er.*,
        COALESCE(ca.unique_customers, 0) as unique_customers,
        COALESCE(ca.premium_customer_percentage, 0) as premium_customer_percentage,
        COALESCE(ca.countries_covered, 0) as countries_covered,
        COALESCE(ca.avg_customer_tenure_days, 0) as avg_customer_tenure_days,
        COALESCE(ca.premium_customers_count, 0) as premium_customers_count,
        -- Комплексная оценка эффективности
        (er.revenue_percentile * 0.4 + 
         (1 - er.seasonality_coefficient) * 0.3 + 
         (er.performance_diff_from_avg / 100) * 0.2 +
         (ca.premium_customer_percentage / 100) * 0.1) as composite_score,
        -- Категоризация сотрудников
        CASE 
            WHEN er.total_revenue > 100000 AND er.roi_ratio > 2 THEN 'Star Performer'
            WHEN er.total_revenue > 50000 AND er.roi_ratio > 1.5 THEN 'High Performer'
            WHEN er.total_revenue > 0 AND er.roi_ratio > 1 THEN 'Solid Performer'
            WHEN er.total_revenue = 0 THEN 'No Sales'
            ELSE 'Needs Improvement'
        END as performance_category,
        -- Рекомендации по развитию
        CASE 
            WHEN er.seasonality_coefficient > 0.5 THEN 'Focus on seasonality management'
            WHEN er.unique_products_sold < 2 THEN 'Expand product knowledge'
            WHEN er.regions_covered < 2 THEN 'Develop regional expertise'
            WHEN ca.premium_customer_percentage < 20 THEN 'Focus on premium clients'
            WHEN er.performance_diff_from_avg < 0 THEN 'Performance coaching needed'
            ELSE 'Continue current strategy'
        END as development_recommendation
    FROM employee_ranked er
    LEFT JOIN customer_analysis ca ON er.emp_id = ca.emp_id
)
-- Финальный результат с фильтрацией топ-10 по композитному score в каждом отделе
SELECT 
    department,
    emp_id,
    name,
    salary,
    performance_score,
    total_revenue,
    roi_ratio,
    revenue_rank,
    performance_rank,
    ROUND(composite_score, 3) as composite_score,
    performance_category,
    development_recommendation,
    unique_customers,
    ROUND(premium_customer_percentage, 1) as premium_customer_pct,
    unique_products_sold,
    regions_covered,
    ROUND(seasonality_coefficient, 3) as seasonality,
    -- Дополнительные метрики для анализа
    ROUND(salary_diff_from_avg, 2) as salary_vs_avg,
    ROUND(performance_diff_from_avg, 2) as performance_vs_avg
FROM final_analysis
QUALIFY ROW_NUMBER() OVER (PARTITION BY department ORDER BY composite_score DESC) <= 10
ORDER BY department, composite_score DESC;
'''

dep_stts = employees_df.alias('e').groupBy('e.department')\
    .agg(
        countDistinct('e.emp_id').alias('total_employees'),
        avg('e.salary').alias('avg_salary'),
        sum('e.salary').alias('total_salary_budget'),
        avg('e.performance_score').alias('avg_performance'))
    
emp_sal_per = employees_df.alias('e').join(sales_df.alias('s'), 'emp_id', 'left')\
    .filter((year(col('s.sale_date')) == 2023) | (col('s.sale_date').isNull()))\
    .groupBy('e.emp_id', 'e.name', 'e.department', 'e.salary', 'e.performance_score', 'e.hire_date')\
    .agg(
        countDistinct('s.transaction_id').alias('total_transactions'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        countDistinct('s.product_category').alias('unique_products_sold'),
        countDistinct('s.region').alias('regions_covered'),
        sum(when(month(col('s.sale_date')).isin([12,1,2]), col('s.amount')).otherwise(0)).alias('winter_revenue'),
        sum(when(month(col('s.sale_date')).isin([3,4,5]), col('s.amount')).otherwise(0)).alias('spring_revenue'),
        sum(when(month(col('s.sale_date')).isin([6,7,8]), col('s.amount')).otherwise(0)).alias('summer_revenue'),
        sum(when(month(col('s.sale_date')).isin([9,10,11]), col('s.amount')).otherwise(0)).alias('fall_revenue'))
    
emp_rank = emp_sal_per.alias('esp').join(dep_stts.alias('ds'), 'department')\
    .select(
        'esp.*',
        col('ds.avg_salary').alias('department_avg_salary'),
        col('ds.avg_performance').alias('department_avg_performance'),
        'ds.total_salary_budget',
        rank().over(Window.partitionBy('esp.department').orderBy(col('esp.total_revenue').desc())).alias('revenue_rank'),
        dense_rank().over(Window.partitionBy('esp.department').orderBy(col('esp.performance_score').desc())).alias('performance_rank'),
        percent_rank().over(Window.partitionBy('esp.department').orderBy('esp.total_revenue')).alias('revenue_percentile'),
        (col('esp.salary') - col('ds.avg_salary')).alias('salary_diff_from_avg'),
        (col('esp.performance_score') - col('ds.avg_performance')).alias('performance_diff_from_avg'),
        when(col('esp.salary') > 0, col('esp.total_revenue') / col('esp.salary')).otherwise(0).alias('roi_ratio'),
        when((col('esp.winter_revenue') + col('esp.spring_revenue') + col('esp.summer_revenue') + col('esp.fall_revenue')) > 0, sqrt(
            (power(col('esp.winter_revenue') - (col('esp.total_revenue') / 4), 2) +
             power(col('esp.spring_revenue') - (col('esp.total_revenue') / 4), 2) +
             power(col('esp.summer_revenue') - (col('esp.total_revenue') / 4), 2) +
             power(col('esp.fall_revenue') - (col('esp.total_revenue') / 4), 2)) / 4
        ) / (col('esp.total_revenue') / 4)).otherwise(0).alias('seasonality_coefficient'))
    
cust_an = sales_df.alias('s').join(customers_df.alias('c'), 'customer_id')\
    .filter(year(col('s.sale_date')) == 2023)\
    .groupBy('s.emp_id')\
    .agg(
        countDistinct('s.customer_id').alias('unique_customers'),
        (avg(col('c.loyalty_tier').isin(['Gold', 'Platinum']).cast('int')) * 100).alias('premium_customer_percentage'),
        countDistinct('c.country').alias('countries_covered'),
        avg(datediff(current_date(), col('c.signup_date'))).alias('avg_customer_tenure_days'),
        countDistinct(
        when(col('c.loyalty_tier').isin('Gold', 'Platinum'), 's.customer_id')).alias('premium_customers_count'))
    
fin_an = emp_rank.alias('er').join(cust_an.alias('ca'), 'emp_id', 'left')\
    .select(
        'er.*',
        coalesce(col('ca.unique_customers'), lit(0)).alias('unique_customers'),
        coalesce(col('ca.premium_customer_percentage'), lit(0)).alias('premium_customer_percentage'),
        coalesce(col('ca.countries_covered'), lit(0)).alias('countries_covered'),
        coalesce(col('ca.avg_customer_tenure_days'), lit(0)).alias('avg_customer_tenure_days'),
        coalesce(col('ca.premium_customers_count'), lit(0)).alias('premium_customers_count'),
        (col('er.revenue_percentile') * 0.4 +
        (1 - col('er.seasonality_coefficient')) * 0.3 +
        (col('er.performance_diff_from_avg') / 100) * 0.2 +
        (col('ca.premium_customer_percentage') / 100) * 0.1).alias('composite_score'),
        when((col('er.total_revenue') > 100000) & (col('er.roi_ratio') > 2), 'Star Performer')\
        .when((col('er.total_revenue') > 50000) & (col('er.roi_ratio') > 1.5), 'High Performer')\
        .when((col('er.total_revenue') > 0) & (col('er.roi_ratio') > 1), 'Solid Performer')\
        .when((col('er.total_revenue') == 0) , 'No Sales')\
        .otherwise('Needs Improvement').alias('performance_category'),
        when(col('er.seasonality_coefficient') > 0.5, 'Focus on seasonality management')\
        .when(col('er.unique_products_sold') < 2, 'Expand product knowledge')\
        .when(col('er.regions_covered') < 2, 'Develop regional expertise')\
        .when(col('ca.premium_customer_percentage') < 20, 'Focus on premium clients')\
        .when(col('er.performance_diff_from_avg') < 0, 'Performance coaching needed')\
        .otherwise('Continue current strategy').alias('development_recommendation'))
    
spark_query = fin_an.withColumn('rn', row_number().over(Window.partitionBy('department').orderBy(col('composite_score').desc())))\
    .filter(col('rn') <= 10)\
    .select(
        'department',
        'emp_id',
        'name',
        'salary',
        'performance_score',
        'total_revenue',
        'roi_ratio',
        'revenue_rank',
        'performance_rank',
        round(col('composite_score'), 3).alias('composite_score'),
        'performance_category',
        'development_recommendation',
        'unique_customers',
        round(col('premium_customer_percentage'), 1).alias('premium_customer_pct'),
        'unique_products_sold',
        'regions_covered',
        round(col('seasonality_coefficient'), 3).alias('seasonality'),
        round(col('salary_diff_from_avg'), 2).alias('salary_vs_avg'),
        round(col('performance_diff_from_avg'), 2).alias('performance_vs_avg')
    ).orderBy('department', col('composite_score').desc())

spark_query.show(1,truncate=False,vertical=True)
spark.sql(sql_query).show(1,truncate=False,vertical=True)